# GLM-OCR Test Notebook

Bu notebook GLM-OCR modelini Google Colab'da (T4 GPU) test eder.

**Pipeline:** PDF/Görüntü → GLM-OCR (GPU) → Metin → Anonimizasyon → Temiz metin

**Gereksinimler:** Colab T4 GPU (ücretsiz tier yeterli)

---

## 1. Kurulum

In [ ]:
# GPU kontrolü
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

In [ ]:
# GLM-OCR ve bağımlılıkları kur
!pip install -q "glmocr[selfhosted]" PyMuPDF pytesseract
!apt-get install -y -q tesseract-ocr tesseract-ocr-tur > /dev/null 2>&1
print("Kurulum tamamlandı!")

## 2. Dosya Yükleme

PDF veya görüntü dosyalarınızı yükleyin.

In [ ]:
import os
from google.colab import files

# Klasör oluştur
os.makedirs("input_files", exist_ok=True)
os.makedirs("output", exist_ok=True)

# Dosya yükle
print("PDF veya görüntü dosyalarınızı yükleyin:")
uploaded = files.upload()

for filename in uploaded:
    with open(f"input_files/{filename}", "wb") as f:
        f.write(uploaded[filename])
    print(f"  ✓ {filename} ({len(uploaded[filename])//1024} KB)")

print(f"\nToplam {len(uploaded)} dosya yüklendi.")

## 3. GLM-OCR ile Metin Çıkarma (GPU)

In [ ]:
import fitz  # PyMuPDF
from PIL import Image
import time
from pathlib import Path
from transformers import AutoModelForVision2Seq, AutoProcessor
import torch
import base64
import io


def pdf_to_images(pdf_path, dpi=300):
    """PDF sayfalarını görüntülere çevir."""
    doc = fitz.open(pdf_path)
    images = []
    for page in doc:
        pix = page.get_pixmap(dpi=dpi)
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        images.append(img)
    doc.close()
    return images


def load_image(file_path):
    """Dosyayı yükle — PDF ise sayfalara böl, görüntüyse direkt aç."""
    ext = Path(file_path).suffix.lower()
    if ext == ".pdf":
        return pdf_to_images(file_path)
    else:
        return [Image.open(file_path).convert("RGB")]


print("Yüklenen dosyalar:")
for f in sorted(os.listdir("input_files")):
    print(f"  - {f}")

In [ ]:
# ── Yöntem 1: vLLM sunucusu ile GLM-OCR ──
# Bu hücre vLLM sunucusunu başlatır (arka planda)

import subprocess, time, requests

print("vLLM sunucusu başlatılıyor (ilk seferde model indirme ~2 dk sürebilir)...")
vllm_proc = subprocess.Popen(
    ["python", "-m", "vllm.entrypoints.openai.api_server",
     "--model", "zai-org/GLM-OCR",
     "--port", "8080",
     "--served-model-name", "glm-ocr",
     "--dtype", "half",
     "--max-model-len", "8192",
     "--gpu-memory-utilization", "0.85"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

# Sunucunun hazır olmasını bekle
for i in range(120):
    try:
        r = requests.get("http://localhost:8080/health", timeout=2)
        if r.status_code == 200:
            print(f"\n✓ vLLM sunucusu hazır! ({i} saniyede)")
            break
    except:
        pass
    if i % 10 == 0:
        print(f"  Bekleniyor... {i}s")
    time.sleep(1)
else:
    print("\n⚠ vLLM başlatılamadı. Loglar:")
    vllm_proc.terminate()
    print(vllm_proc.stdout.read().decode()[-2000:])

In [ ]:
# ── GLM-OCR ile OCR ──

import requests, base64, io, json

VLLM_URL = "http://localhost:8080/v1/chat/completions"


def glm_ocr_single(image: Image.Image) -> str:
    """Tek bir görüntüyü GLM-OCR ile OCR yap."""
    buf = io.BytesIO()
    image.save(buf, format="PNG")
    b64 = base64.b64encode(buf.getvalue()).decode()

    resp = requests.post(VLLM_URL, json={
        "model": "glm-ocr",
        "messages": [{
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
                {"type": "text", "text": "OCR"}
            ]
        }],
        "max_tokens": 4096,
        "temperature": 0
    }, timeout=300)

    data = resp.json()
    return data["choices"][0]["message"]["content"]


def glm_ocr_file(file_path: str) -> str:
    """Dosyayı (PDF veya görüntü) GLM-OCR ile OCR yap."""
    images = load_image(file_path)
    all_text = []
    for i, img in enumerate(images):
        print(f"  Sayfa {i+1}/{len(images)}...", end=" ")
        t0 = time.time()
        text = glm_ocr_single(img)
        print(f"{time.time()-t0:.1f}s")
        all_text.append(text)
    return "\n\n".join(all_text)


# Tüm dosyaları işle
glm_results = {}
for filename in sorted(os.listdir("input_files")):
    filepath = f"input_files/{filename}"
    print(f"\n{'='*50}")
    print(f"GLM-OCR: {filename}")
    print("="*50)
    t0 = time.time()
    text = glm_ocr_file(filepath)
    elapsed = time.time() - t0
    glm_results[filename] = text

    # Kaydet
    stem = Path(filename).stem
    with open(f"output/{stem}_glmocr.txt", "w", encoding="utf-8") as f:
        f.write(text)

    print(f"\n--- Çıktı ({elapsed:.1f}s, {len(text)} karakter) ---")
    print(text[:500])
    print("...")

## 4. Tesseract OCR ile Karşılaştırma

In [ ]:
import pytesseract


def tesseract_ocr_file(file_path: str) -> str:
    """Dosyayı Tesseract ile OCR yap."""
    images = load_image(file_path)
    all_text = []
    for i, img in enumerate(images):
        text = pytesseract.image_to_string(img, lang="tur", config="--psm 6")
        all_text.append(text)
    return "\n\n".join(all_text)


# Tüm dosyaları Tesseract ile işle
tess_results = {}
for filename in sorted(os.listdir("input_files")):
    filepath = f"input_files/{filename}"
    print(f"Tesseract: {filename}...", end=" ")
    t0 = time.time()
    text = tesseract_ocr_file(filepath)
    elapsed = time.time() - t0
    tess_results[filename] = text

    stem = Path(filename).stem
    with open(f"output/{stem}_tesseract.txt", "w", encoding="utf-8") as f:
        f.write(text)

    print(f"{elapsed:.1f}s, {len(text)} karakter")

## 5. Karşılaştırma Tablosu

In [ ]:
from IPython.display import HTML

# Karşılaştırma tablosu
rows = []
for filename in sorted(glm_results.keys()):
    glm_text = glm_results.get(filename, "")
    tess_text = tess_results.get(filename, "")

    # Basit kalite metrikleri
    glm_words = len(glm_text.split())
    tess_words = len(tess_text.split())

    # Türkçe karakter oranı (OCR kalite göstergesi)
    tr_chars = set("çğıöşüÇĞİÖŞÜ")
    glm_tr = sum(1 for c in glm_text if c in tr_chars)
    tess_tr = sum(1 for c in tess_text if c in tr_chars)

    rows.append({
        "Dosya": filename[:30],
        "GLM-OCR (kelime)": glm_words,
        "Tesseract (kelime)": tess_words,
        "GLM Türkçe karakter": glm_tr,
        "Tess Türkçe karakter": tess_tr,
    })

# Tablo
html = "<table border='1' style='border-collapse:collapse; font-size:14px;'>"
html += "<tr>" + "".join(f"<th style='padding:8px; background:#f0f0f0;'>{k}</th>" for k in rows[0].keys()) + "</tr>"
for row in rows:
    html += "<tr>" + "".join(f"<td style='padding:8px;'>{v}</td>" for v in row.values()) + "</tr>"
html += "</table>"
display(HTML(html))

In [ ]:
# Yan yana metin karşılaştırması (ilk dosya)
from IPython.display import HTML

first_file = sorted(glm_results.keys())[0]
glm_text = glm_results[first_file]
tess_text = tess_results[first_file]

html = f"""
<h3>{first_file}</h3>
<div style='display:flex; gap:20px;'>
  <div style='flex:1; border:1px solid #ccc; padding:10px; max-height:600px; overflow:auto;'>
    <h4 style='color:green;'>GLM-OCR</h4>
    <pre style='white-space:pre-wrap; font-size:12px;'>{glm_text[:3000]}</pre>
  </div>
  <div style='flex:1; border:1px solid #ccc; padding:10px; max-height:600px; overflow:auto;'>
    <h4 style='color:blue;'>Tesseract</h4>
    <pre style='white-space:pre-wrap; font-size:12px;'>{tess_text[:3000]}</pre>
  </div>
</div>
"""
display(HTML(html))

## 6. Çıktıları İndir

In [ ]:
# Tüm çıktıları zip yapıp indir
import shutil
shutil.make_archive("ocr_results", "zip", "output")
files.download("ocr_results.zip")
print("\n✓ Tüm OCR sonuçları indirildi!")

## 7. (Opsiyonel) glmocr Kütüphanesi ile Test

Eğer vLLM sunucusu çalışıyorsa, `glmocr` kütüphanesini de deneyebilirsiniz.
Bu layout detection + paralel OCR yaparak daha iyi sonuç verebilir.

In [ ]:
# glmocr kütüphanesi ile (layout detection dahil)
import yaml

# Config dosyası oluştur
config = {
    "pipeline": {
        "maas": {"enabled": False},
        "ocr_api": {
            "api_host": "localhost",
            "api_port": 8080,
        }
    }
}
with open("glmocr_config.yaml", "w") as f:
    yaml.dump(config, f)

from glmocr import GlmOcr

parser = GlmOcr(config_path="glmocr_config.yaml", layout_device="cuda", log_level="INFO")

for filename in sorted(os.listdir("input_files")):
    filepath = f"input_files/{filename}"
    print(f"\nglmocr: {filename}")
    t0 = time.time()
    result = parser.parse(filepath)
    elapsed = time.time() - t0

    # Sonucu kaydet
    result.save(output_dir=f"output/glmocr_layout/")
    print(f"  {elapsed:.1f}s")
    if hasattr(result, 'markdown'):
        print(result.markdown[:500])

## 8. Temizlik

In [ ]:
# vLLM sunucusunu durdur
try:
    vllm_proc.terminate()
    print("vLLM sunucusu durduruldu.")
except:
    print("vLLM zaten durmuş.")